# DNA Linker Prediction Pipeline - Example Notebook

This notebook demonstrates the complete DNA Linker prediction pipeline using optimized vectorized functions. The pipeline identifies potential linker connections between nucleosome particles based on spatial proximity and directional compatibility.

## Installation

Before running this notebook, ensure the dna_linker package is installed:

# Install the package in editable mode
pip install -e ..

Or add the package directory to your PYTHONPATH (done automatically in the setup cell below).

In [ ]:
# Setup: Add dna_linker to Python path
import sys
import os
repo_root = os.path.abspath("../dna_linker")
sys.path.insert(0, repo_root)
print(f"Added {repo_root} to sys.path")


## 1. Configuration and Imports

First, we import all required modules and set up the configuration parameters.

In [ ]:
# Standard library imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import time
from pathlib import Path

# CryoCAT imports for handling .em and .mrc files
from cryocat import cryomotl
from cryocat import cryomap

# DNA Linker module imports
from dna_linker import config
from dna_linker.dna_linkers import (
    calculate_probabilities_all_connexions,
    calculate_linker_length_connected,
    create_connection,
    find_next_maximum,
    write_out_EmMolt_linker,
    save_connnections
)
from dna_linker.utils_motlFiles import (
    create_shifted_motif_lists_along_linker
)

# Display settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("All imports successful!")

## 2. Configuration Parameters

The following parameters control the pipeline behavior:

- **tracing_distance**: Maximum distance (in voxels) for initial particle tracing
- **lo**: Persistence length parameter (in voxels) for bending energy calculation
- **pmin**: Minimum probability threshold for considering a connection valid
- **pixel_size**: Voxel size in nm

In [ ]:
# Pipeline configuration parameters
tracing_distance = config.tracing_distance  # 350 voxels
lo = config.lo  # Persistence length parameter
pmin = config.pmin  # Minimum probability threshold: 0.1
pixel_size = config.pixel_size  # 1 voxel per unit
bin_factor = config.bin  # Binning factor: 1.0

# Calculate max distance for tracing
max_distance = tracing_distance / (pixel_size * bin_factor)

print("Pipeline Configuration:")
print(f"  Tracing distance: {tracing_distance} voxels")
print(f"  Persistence length (lo): {lo:.2f} voxels")
print(f"  Min probability threshold: {pmin}")
print(f"  Max tracing distance: {max_distance} voxels")

## 3. Define Paths

Set up input and output paths for the test dataset.

In [ ]:
# Input paths
INPUT_DIR = Path("../dna_linker/inputs")

MOTL_FILE = INPUT_DIR / "motl_EMD2601_dropped_01.em"
ENTRY_MASK = INPUT_DIR / "Threshold_ref_entrymask_r2_resamp_righthand.mrc"
EXIT_MASK = INPUT_DIR / "Threshold_ref_exitmask_r2_resamp_righthand.mrc"
ORIGIN_ENTRY_MASK = INPUT_DIR / "Threshold_ref_Origin_entrymask_r2_resamp_righthand.mrc"
ORIGIN_EXIT_MASK = INPUT_DIR / "Threshold_ref_Origin_exitmask_r2_resamp_righthand.mrc"

# Output base directory
OUTPUT_BASE = Path("../dna_linker/outputs/example_notebook")
OUTPUT_CLUSTERS = OUTPUT_BASE / "clusters_20nm"
OUTPUT_LINKERS = OUTPUT_BASE / "A_linkers_20nm"
OUTPUT_DICTIONARY = OUTPUT_BASE / "A_Connections_dictionary_20nm"

# Create output directories
for path in [OUTPUT_CLUSTERS, OUTPUT_LINKERS, OUTPUT_DICTIONARY]:
    path.mkdir(parents=True, exist_ok=True)

print("Input files:")
print(f"  MOTL file: {MOTL_FILE.name}")
print(f"  Entry mask: {ENTRY_MASK.name}")
print(f"  Exit mask: {EXIT_MASK.name}")
print(f"\nOutput directories created:")
print(f"  {OUTPUT_BASE}")
print(f"  {OUTPUT_CLUSTERS}")
print(f"  {OUTPUT_LINKERS}")
print(f"  {OUTPUT_DICTIONARY}")

## 4. Step 1: Load MOTL Data

Load the particle coordinate data from the .em file using CryoCAT.

In [ ]:
# Step 1: Load MOTL data
print("=" * 60)
print("STEP 1: Loading MOTL Data")
print("=" * 60)

start_time = time.time()

# Load the full MOTL file
motl_all = cryomotl.EmMotl(input_motl=str(MOTL_FILE))

elapsed = time.time() - start_time

print(f"Loaded MOTL file: {MOTL_FILE.name}")
print(f"Total particles: {len(motl_all.df)}")
print(f"Tomogram IDs: {motl_all.df['tomo_id'].unique()}")
print(f"Time: {elapsed:.3f} seconds")

# Display first few rows
print("\nMOTL DataFrame preview:")
display(motl_all.df.head())

## 5. Step 2: Apply Entry/Exit Masks

Create shifted motif lists by applying entry and exit masks. This recenters particles to the DNA entry/exit points on the nucleosome surface.

In [ ]:
# Step 2: Apply Entry/Exit Masks
print("=" * 60)
print("STEP 2: Applying Entry/Exit Masks")
print("=" * 60)

start_time = time.time()

# Create shifted motif lists along the linker
output_paths = create_shifted_motif_lists_along_linker(
    motl_trace_input=str(MOTL_FILE),
    output_path_cluster=str(OUTPUT_CLUSTERS) + "/",
    path_mask=str(INPUT_DIR) + "/",
    origin_entry=ORIGIN_ENTRY_MASK.name,
    entry=ENTRY_MASK.name,
    origin_exit=ORIGIN_EXIT_MASK.name,
    exit=EXIT_MASK.name,
    prefix="All_"
)

elapsed = time.time() - start_time

print(f"\nCreated shifted MOTL files:")
for path in output_paths:
    print(f"  {Path(path).name}")
print(f"Time: {elapsed:.3f} seconds")

In [ ]:
# Load the shifted MOTL files
print("Loading shifted MOTL files...")

# Load entry and exit motif lists
motl_entry = cryomotl.EmMotl(input_motl=str(OUTPUT_CLUSTERS / "All_motl_entry.em"))
motl_entry2 = cryomotl.EmMotl(input_motl=str(OUTPUT_CLUSTERS / "All_motl_entry2.em"))
motl_exit = cryomotl.EmMotl(input_motl=str(OUTPUT_CLUSTERS / "All_motl_exit.em"))
motl_exit2 = cryomotl.EmMotl(input_motl=str(OUTPUT_CLUSTERS / "All_motl_exit2.em"))

print(f"Entry particles: {len(motl_entry.df)}")
print(f"Entry2 particles: {len(motl_entry2.df)}")
print(f"Exit particles: {len(motl_exit.df)}")
print(f"Exit2 particles: {len(motl_exit2.df)}")

## 6. Step 3: Compute Probability Matrix (Vectorized)

This is the **key optimization** in the pipeline. Instead of computing probabilities one pair at a time, we use NumPy broadcasting to compute all pairwise probabilities simultaneously.

The probability model considers:
- **Distance probability**: Exponential decay based on linker length (exp(-L/lo))
- **Bending probability**: Energy penalty based on angular deviation between particle orientations

In [ ]:
# Step 3: Compute Probability Matrix (Vectorized)
print("=" * 60)
print("STEP 3: Computing Probability Matrix (Vectorized)")
print("=" * 60)

start_time = time.time()

# Use the optimized vectorized probability calculation
# This computes all N×N×4 probabilities in a single vectorized operation
probs = calculate_probabilities_all_connexions(
    motl=motl_all,
    motl_exit=motl_exit,
    motl_exit2=motl_exit2,
    motl_entry=motl_entry,
    motl_entry2=motl_entry2,
    lo=lo
)

elapsed = time.time() - start_time

print(f"\nProbability matrix shape: {probs.shape}")
print(f"  Dimension 0: {probs.shape[0]} particles (source)")
print(f"  Dimension 1: {probs.shape[1]} particles (target)")
print(f"  Dimension 2: 4 cases (entry-entry, exit-entry, entry-exit, exit-exit)")
print(f"\nProbability statistics per case:")
case_names = ["entry-entry", "exit-entry", "entry-exit", "exit-exit"]
for i, name in enumerate(case_names):
    case_probs = probs[:, :, i]
    print(f"  {name}: min={case_probs.min():.4f}, max={case_probs.max():.4f}, mean={case_probs.mean():.4f}")
print(f"\nTime: {elapsed:.3f} seconds")

### Visualization: Max Probability Across All Connection Cases

This visualization shows the maximum probability across all connection types (entry-entry, entry-exit, exit-entry, exit-exit) for each particle pair.

In [ ]:
# Visualization: Max Probability Across All Cases
fig, ax = plt.subplots(figsize=(10, 8))

# Take max across all connection types
max_probs = np.max(probs, axis=2)

im = ax.imshow(max_probs, cmap='viridis', aspect='auto')
ax.set_xlabel('Particle Index')
ax.set_ylabel('Particle Index')
ax.set_title('Max Probability: All Connection Types')
plt.colorbar(im, ax=ax, label='Max Probability')

plt.tight_layout()
plt.savefig(str(OUTPUT_BASE / 'max_probability_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved: {OUTPUT_BASE / 'max_probability_heatmap.png'}")

## 7. Step 4: Find Connected Particles

Use a greedy algorithm to find the most probable connections. At each step, we select the highest probability connection and remove the involved particles from further consideration.

**Note on remaining_connections**: Each particle can have up to 2 connections (DNA enters and exits each nucleosome).

In [ ]:
# Step 4: Find Connected Particles (Greedy Algorithm)
print("=" * 60)
print("STEP 4: Finding Connected Particles")
print("=" * 60)

start_time = time.time()

# Initialize connections dictionary and remaining connections tracker
connections = {i: [] for i in range(len(motl_all.df))}
remaining_connections = {i: 2 for i in range(len(motl_all.df))}  # Each particle can have 2 connections

# Find connections using greedy algorithm
# Stop when all remaining particles have 0 connections or probability drops below threshold
iteration = 0
max_iterations = len(motl_all.df) * 2  # Maximum possible connections

while iteration < max_iterations:
    # Find the maximum probability in the probability matrix
    max_index = find_next_maximum(probs)
    max_prob = probs[max_index]
    
    # Stop if probability is below threshold
    if max_prob < pmin:
        break
    
    # Create connection
    connections, remaining_connections = create_connection(
        connections, probs, max_index, remaining_connections
    )
    
    # Zero out the used connections to avoid reuse
    i, j, case = max_index
    probs[i, j, :] = 0
    probs[j, i, :] = 0
    
    iteration += 1

elapsed = time.time() - start_time

# Count total connections
total_connections = sum(len(conns) for conns in connections.values()) // 2
particles_with_connections = sum(1 for conns in connections.values() if len(conns) > 0)

print(f"Greedy algorithm completed:")
print(f"  Total iterations: {iteration}")
print(f"  Total connections found: {total_connections}")
print(f"  Particles with connections: {particles_with_connections}/{len(motl_all.df)}")
print(f"\nTime: {elapsed:.3f} seconds")

In [ ]:
# Analyze connections by case
print("Connections breakdown by case:")
case_counts = {0: 0, 1: 0, 2: 0, 3: 0}
case_probs = {0: [], 1: [], 2: [], 3: []}

for i, conns in connections.items():
    for conn in conns:
        j, prob, case = conn
        case_counts[case] += 1
        case_probs[case].append(prob)

case_names = ["entry-entry", "exit-entry", "entry-exit", "exit-exit"]
for i, name in enumerate(case_names):
    if case_counts[i] > 0:
        avg_prob = np.mean(case_probs[i]) if case_probs[i] else 0
        print(f"  {name}: {case_counts[i]} connections, avg prob: {avg_prob:.4f}")
    else:
        print(f"  {name}: 0 connections")

## 8. Step 5: Compute Linker Lengths

Calculate the linker lengths (Euclidean distances) for all connected particle pairs.

In [ ]:
# Step 5: Compute Linker Lengths
print("=" * 60)
print("STEP 5: Computing Linker Lengths")
print("=" * 60)

start_time = time.time()

# Calculate linker lengths for connected particles
linker_lengths = calculate_linker_length_connected(
    connections=connections,
    motl_exit=motl_exit,
    motl_entry=motl_entry,
    p_min=pmin
)

elapsed = time.time() - start_time

if len(linker_lengths) > 0:
    print(f"\nLinker length statistics:")
    print(f"  Number of linkers: {len(linker_lengths)}")
    print(f"  Mean length: {linker_lengths.mean():.2f} voxels")
    print(f"  Std length: {linker_lengths.std():.2f} voxels")
    print(f"  Min length: {linker_lengths.min():.2f} voxels")
    print(f"  Max length: {linker_lengths.max():.2f} voxels")
else:
    print("No linkers found above probability threshold.")

print(f"\nTime: {elapsed:.3f} seconds")

## 9. Step 6: Generate Outputs

Save the results to files:
- **Connections dictionary**: Pickle file with all connection data
- **Linker MOTL**: .em file with linker coordinates
- **Visualizations**: Probability and linker length plots

In [ ]:
# Step 6: Generate Outputs
print("=" * 60)
print("STEP 6: Generating Outputs")
print("=" * 60)

start_time = time.time()

# Save connections dictionary
connections_file = OUTPUT_DICTIONARY / "Connectivity_example.pickle"
save_connnections(connections, str(connections_file))
print(f"Saved connections to: {connections_file}")

# Write linker MOTL file
linker_output = OUTPUT_LINKERS / "motl_example_linkers.em"
write_out_EmMolt_linker(
    connections=connections,
    motl_exit=motl_exit,
    motl_entry=motl_entry,
    motl_exit2=motl_exit2,
    motl_entry2=motl_entry2,
    output_filename=str(linker_output),
    min_prob=pmin
)
print(f"Saved linker MOTL to: {linker_output}")

elapsed = time.time() - start_time
print(f"\nTime: {elapsed:.3f} seconds")

## 10. Visualizations

In [ ]:
# Visualization 1: Probability Distribution by Case
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
case_names = ["Entry-Entry", "Exit-Entry", "Entry-Exit", "Exit-Exit"]

for idx, (ax, name) in enumerate(zip(axes.flat, case_names)):
    probs_case = probs[:, :, idx].flatten()
    probs_case = probs_case[probs_case > 0]  # Filter out zeros
    
    ax.hist(probs_case, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(pmin, color='red', linestyle='--', label=f'Threshold: {pmin}')
    ax.set_xlabel('Probability')
    ax.set_ylabel('Count')
    ax.set_title(f'{name} (n={len(probs_case)})')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Probability Distribution by Connection Case', fontsize=14)
plt.tight_layout()
plt.savefig(str(OUTPUT_BASE / 'probability_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved: {OUTPUT_BASE / 'probability_distributions.png'}")

In [ ]:
# Visualization 2: Linker Length Distribution
if len(linker_lengths) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(linker_lengths, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].axvline(linker_lengths.mean(), color='red', linestyle='--', 
                    label=f'Mean: {linker_lengths.mean():.1f}')
    axes[0].set_xlabel('Linker Length (voxels)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Distribution of Linker Lengths')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Box plot with individual points
    axes[1].boxplot(linker_lengths, vert=True)
    axes[1].scatter(np.ones(len(linker_lengths)), linker_lengths, alpha=0.5, color='steelblue')
    axes[1].set_ylabel('Linker Length (voxels)')
    axes[1].set_title('Linker Length Box Plot')
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle('Linker Length Analysis', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_BASE / 'linker_lengths.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Saved: {OUTPUT_BASE / 'linker_lengths.png'}")

In [ ]:
# Visualization 3: Probability Heatmap (Entry-Entry Case)
fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(probs[:, :, 0], cmap='viridis', aspect='auto')
ax.set_xlabel('Particle Index')
ax.set_ylabel('Particle Index')
ax.set_title('Probability Matrix: Entry-Entry Connections')
plt.colorbar(im, ax=ax, label='Probability')

plt.tight_layout()
plt.savefig(str(OUTPUT_BASE / 'probability_heatmap_entry_entry.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved: {OUTPUT_BASE / 'probability_heatmap_entry_entry.png'}")

## 11. Summary and Runtime

In [ ]:
# Summary
print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)

print("\nInput Configuration:")
print(f"  MOTL file: {MOTL_FILE.name}")
print(f"  Entry mask: {ENTRY_MASK.name}")
print(f"  Exit mask: {EXIT_MASK.name}")

print("\nParameters:")
print(f"  Tracing distance: {tracing_distance} voxels")
print(f"  Persistence length (lo): {lo:.2f} voxels")
print(f"  Min probability: {pmin}")

print("\nResults:")
print(f"  Total particles: {len(motl_all.df)}")
print(f"  Particles with connections: {particles_with_connections}")
print(f"  Total connections: {total_connections}")
if len(linker_lengths) > 0:
    print(f"  Linker lengths: mean={linker_lengths.mean():.2f} ± {linker_lengths.std():.2f} voxels")

print("\nOutput Files:")
print(f"  {OUTPUT_DICTIONARY / 'Connectivity_example.pickle'}")
print(f"  {OUTPUT_LINKERS / 'motl_example_linkers.em'}")
print(f"  {OUTPUT_BASE / 'probability_distributions.png'}")
print(f"  {OUTPUT_BASE / 'linker_lengths.png'}")
print(f"  {OUTPUT_BASE / 'probability_heatmap_entry_entry.png'}")

print("\n" + "=" * 60)
print("Pipeline completed successfully!")
print("=" * 60)